In [13]:
!pip install transformers datasets 

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.is_available())

PyTorch: 2.11.0
GPU: False


In [5]:
texts = [
    "这部电影真好看", "演员演技很棒", "剧情精彩", "非常推荐", "值得一看",
    "画面很美", "导演功力深厚", "故事感人", "特效震撼", "音乐动听",
    "太差了，浪费钱", "剧情无聊", "演员演技尴尬", "烂片", "后悔买票",
    "不知所云", "节奏拖沓", "特效五毛", "毫无逻辑", "浪费时间"
]
labels = [1]*10 + [0]*10  # 1=正面, 0=负面

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)
print(f"训练: {len(train_texts)}, 验证: {len(val_texts)}")

训练: 16, 验证: 4


In [7]:
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')

# 新版：直接调用 tokenizer()，不用 encode_plus
sample_text = "这部电影真好看"
encoding = tokenizer(
    sample_text,
    add_special_tokens=True,
    max_length=20,
    padding='max_length',
    truncation=True,
    
    return_tensors='pt'
)

print("input_ids:", encoding['input_ids'].flatten().tolist())
print("attention_mask:", encoding['attention_mask'].flatten().tolist())
print("tokens:", tokenizer.convert_ids_to_tokens(encoding['input_ids'].flatten().tolist()))

input_ids: [101, 6821, 6956, 4510, 2512, 4696, 1962, 4692, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
tokens: ['[CLS]', '这', '部', '电', '影', '真', '好', '看', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [12]:
class BERTDataset(Dataset):
    def __init__(self,texts,labels,tokenizer,max_len=64):
        self.texts=texts
        self.labels=labels
        self.tokenizer=tokenizer
        self.max_len=max_len
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self,idx):
        text=self.texts[idx]
        label=self.labels[idx]
        #新版调用方式
        encoding=self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':encoding['input_ids'].flatten(),
            'attention_mask':encoding['attention_mask'].flatten(),
            'label':torch.tensor(label,dtype=torch.float)
        }
train_dataset=BERTDataset(train_texts,train_labels,tokenizer)
val_dataset=BERTDataset(val_texts,val_labels,tokenizer)
train_loader=DataLoader(train_dataset,batch_size=4,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=4)
batch=next(iter(train_loader))
print("input_ids:",batch["input_ids"].shape)
print("attention_mask:",batch['attention_mask'].shape)
print("label:",batch['label'].shape)


input_ids: torch.Size([4, 64])
attention_mask: torch.Size([4, 64])
label: torch.Size([4])


In [13]:
model=BertForSequenceClassification.from_pretrained(
    'bert-base-chinese',
    num_labels=1
)

print("分类头:",model.classifier)
print("总参数:",sum(p.numel()for p in model.parameters())/1e6,"M")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-chinese
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


分类头: Linear(in_features=768, out_features=1, bias=True)
总参数: 102.268417 M


In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# 新版 AdamW 从 torch.optim 导入
optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

epochs = 20

for epoch in range(epochs):
    # 训练
    model.train()   #模型切换到训练模式s
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.squeeze()
        
        loss = criterion(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    # 验证
    model.eval()
    val_loss = 0
    val_acc = 0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            loss = criterion(logits, labels)
            
            preds = (logits > 0).float()
            acc = (preds == labels).float().mean()
            
            val_loss += loss.item()
            val_acc += acc.item()
    
    print(f"Epoch {epoch+1:2d} | Train Loss={total_loss/len(train_loader):.4f} | "
          f"Val Loss={val_loss/len(val_loader):.4f} | Val Acc={val_acc/len(val_loader):.4f}")

Epoch 1|Train Loss=0.6908|Val Loss=0.7112|Val Acc=0.7500
Epoch 2|Train Loss=0.6477|Val Loss=0.7025|Val Acc=0.5000
Epoch 3|Train Loss=0.5876|Val Loss=0.7580|Val Acc=0.5000
Epoch 4|Train Loss=0.4853|Val Loss=0.8192|Val Acc=0.5000
Epoch 5|Train Loss=0.4809|Val Loss=0.6998|Val Acc=0.5000
Epoch 6|Train Loss=0.3967|Val Loss=0.6787|Val Acc=0.5000
Epoch 7|Train Loss=0.3240|Val Loss=0.6595|Val Acc=0.5000
Epoch 8|Train Loss=0.2759|Val Loss=0.6250|Val Acc=0.5000
Epoch 9|Train Loss=0.2543|Val Loss=0.5846|Val Acc=0.7500
Epoch10|Train Loss=0.2141|Val Loss=0.5612|Val Acc=0.7500
Epoch11|Train Loss=0.1831|Val Loss=0.5847|Val Acc=0.7500
Epoch12|Train Loss=0.1622|Val Loss=0.6192|Val Acc=0.7500
Epoch13|Train Loss=0.1310|Val Loss=0.6391|Val Acc=0.7500
Epoch14|Train Loss=0.1211|Val Loss=0.6532|Val Acc=0.5000
Epoch15|Train Loss=0.1057|Val Loss=0.6917|Val Acc=0.5000
Epoch16|Train Loss=0.0930|Val Loss=0.6964|Val Acc=0.5000
Epoch17|Train Loss=0.0724|Val Loss=0.7577|Val Acc=0.5000
Epoch18|Train Loss=0.0700|Val L

In [15]:
model.eval()

def predict(text):
    encoding=tokenizer(text,
                      add_special_tokens=True,
                      max_length=64,
                      padding='max_length',
                      truncation=True,
                      return_tensors='pt')

    input_ids=encoding['input_ids'].to(device)
    attention_mask=encoding['attention_mask'].to(device)

    with torch.no_grad():
        output=model(input_ids=input_ids,attention_mask=attention_mask)
        logit=output.logits.item()
        prob=torch.sigmoid(torch.tensor(logit)).item()
        label="正面"if logit>0 else"负面"

    print(f"文本：{text}")
    print(f"预测：{label},置信度：{prob:.4f}\n")

predict("这部电影太精彩了")
predict("完全看不下去，烂片")
predict("演员表演还行，剧情一般")

文本：这部电影太精彩了
预测：正面,置信度：0.9317

文本：完全看不下去，烂片
预测：负面,置信度：0.0411

文本：演员表演还行，剧情一般
预测：负面,置信度：0.2213

